Libraries

In [1]:
!pip -q install langchain
!pip -q install langchain-community
!pip -q install langchain-text-splitters
!pip -q install chromadb
!pip -q install sentence-transformers
!pip -q install transformers
!pip -q install pypdf
!pip -q install tiktoken
!pip -q install accelerate
!pip -q install bitsandbytes
!pip -q install google-generativeai
!pip install flask pyngrok

Import these Libraries

In [2]:
import os
from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

/tmp/ipykernel_1401/2641019122.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Uploading the PDF Files

In [3]:
uploaded = files.upload()

Saving s10462-024-11082-w.pdf to s10462-024-11082-w.pdf
Saving s10462-025-11346-z.pdf to s10462-025-11346-z.pdf
Saving s40537-024-00957-y.pdf to s40537-024-00957-y.pdf
Saving s40537-025-01323-2.pdf to s40537-025-01323-2.pdf
Saving sensors-25-01153-v3.pdf to sensors-25-01153-v3.pdf


In [4]:
documents = []

for filename in uploaded.keys():
    loader = PyPDFLoader(filename)
    docs = loader.load()
    documents.extend(docs)

print(f"Loaded {len(documents)} pages.")

Loaded 238 pages.


Splitting the Document

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 1012


Embeddings

In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_1401/1869530954.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chroma DB Creation

Retrieval

In [7]:
from langchain_community.vectorstores import Chroma

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="cybersecurity_db"
)

print("Vector database created successfully!")

Vector database created successfully!


In [8]:
retriever = vector_db.as_retriever(search_kwargs={"k":3})

query = "What are the challenges of AI in cybersecurity?"

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content[:700])


--- Result 1 ---
complex problem-solving domains such as cybersecurity, where it addresses sophisti -
cated cyber threats. This transformative technology continues to push the boundaries of 
what machines are capable of, aiming to enhance human capabilities and automate tasks 
through assisted, augmented, and autonomous intelligence [15].
The use of AI in cybersecurity is increasingly critical due to its capacity to analyze 
vast amounts of data rapidly, detect patterns, and identify potential threats with high 
efficiency. In a digital era characterized by ever-evolving cyber threats, traditional secu -
rity measures often fall short in both the speed and sophistication needed to counter -
act modern cybera

--- Result 2 ---
mentioned. In [38] a systematic review of AI applications in cybersecurity categorizes 
236 studies within the NIST framework. It highlights AI’s role in automating tasks, 
enhancing threat detection, and improving response accuracy using ML, DL, natural 
languag

Installing Gemini

In [9]:
!pip install -q google-generativeai langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00


In [10]:
from google.colab import userdata
import google.generativeai as genai

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [11]:
model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content(
    "Explain Retrieval-Augmented Generation in two sentences."
)

print(response.text)

Retrieval-Augmented Generation (RAG) is an AI framework that enhances the reliability of large language models (LLMs) by giving them access to external, up-to-date information. Before generating a response, the system first retrieves relevant documents from a knowledge base and then uses this retrieved context to ground the LLM's answer, reducing hallucinations and improving factual accuracy.


In [12]:
def research_agent(question):
  # to Retrieve Relevent chunks
    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    # Decision making
    if "summarize" in question.lower():
        prompt = f"""
You are an AI Research Assistant.

Summarize the following cybersecurity research in approximately 150 words.

Context:
{context}
"""

    else:
        prompt = f"""
You are a Cybersecurity Research Assistant.

Use ONLY the provided context.

If the answer is not found in the context, say:
'I could not find this information in the uploaded research papers.'

Context:
{context}

Question:
{question}
"""

    response = model.generate_content(prompt)

    print("\n📚 Sources Used:")
    for i, doc in enumerate(docs):
        print(f"{i+1}. {doc.metadata.get('source', 'Unknown')}")

    return response.text

In [13]:
while True:
    question = input("\nAsk a question (type 'exit' to quit): ")

    if question.lower() == "exit":
        break

    answer = research_agent(question)

    print("\n Answer:")
    print(answer)


Ask a question (type 'exit' to quit): iwhat is the scope of ai in cybersecurity

📚 Sources Used:
1. s40537-024-00957-y.pdf
2. s40537-024-00957-y.pdf
3. s40537-024-00957-y.pdf

 Answer:
The scope of AI in cybersecurity includes addressing sophisticated cyber threats, analyzing vast amounts of data rapidly, detecting patterns, and identifying potential threats with high efficiency. It enhances human capabilities and automates tasks through assisted, augmented, and autonomous intelligence.

Specifically, AI's role involves:
*   **Automating tasks**
*   **Enhancing threat detection** (including zero-day threats, malware, phishing, APTs, and insider threats)
*   **Improving response accuracy**
*   **Analyzing potential vulnerabilities proactively**
*   **Real-time monitoring**
*   **Automated responses**
*   **Continuous learning** to adapt to new threats.

Key areas where AI is applied include:
*   Asset management
*   Threat hunting
*   Vulnerability assessment
*   Incident response
*   

In [14]:
question = "What are the challenges of AI in cybersecurity?"

answer = research_agent(question)

print(answer)


📚 Sources Used:
1. s40537-024-00957-y.pdf
2. s40537-024-00957-y.pdf
3. s40537-024-00957-y.pdf
The challenges of AI in cybersecurity mentioned include:
*   practical deployment challenges
*   ethical implications
*   potential biases in AI models
*   the need for standardized benchmarks and evaluation metrics for AI effectiveness
